In [1]:
from datasets import load_dataset

/export/home/ldedieu/CU2/encoder-baseline/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import os

# Set the HF_TOKEN environment variable before running (do NOT commit a token).
ds = load_dataset("HealthDataHub/PARHAF", token=os.environ["HF_TOKEN"])

coding_patients = []
for patient in ds["train"]:
    if patient["pool"]=="CU 2 - ICD-10 coding":
        coding_patients.append(patient)
    
print(len(coding_patients))


'''
patient = ds["train"][0]       # First patient

pool = patient["pool"]

patient_id = patient["id"]
specialty = patient["specialty"]
scenario = patient["suggested_scenario"]
patient_fake_name = scenario["name"]
primary_diagnosis = scenario["primary_diagnosis"]
patient_reports = patient["documents"]
...

# In Hugging Face dataset, sequences of dictionaries are represented 
# as a dictionary of lists (one list per field) rather than a list of dictionaries.
types = patient_reports["type"]
texts = patient_reports["text"]
for type, text in zip(types, texts):
    print(type, text)
'''

In [3]:
coding_patients[0]

{'id': 'CHIR-ORTHO-ET-TRAUMATO-00024',
 'local_id': '00024',
 'specialty': 'CHIR ORTHO ET TRAUMATO',
 'author': 'MSA',
 'reviewer': 'TRI',
 'pool': 'CU 2 - ICD-10 coding',
 'suggested_scenario': {'name': 'Gérard Danard',
  'age': {'value': 79, 'unit': 'ans'},
  'sex': 'F',
  'admission_mode': 'domicile',
  'discharge_mode': 'retour à domicile',
  'primary_procedure': None,
  'primary_diagnosis': {'code': ['M6534'],
   'description': ['Doigt "à ressort" - Main']},
  'type_of_care': 'Chirurgies main, poignet, en ambulatoire'},
 'documents': {'type': ['CRC', 'CRO', 'CRH'],
  'header': ['Compte rendu de consultation de Chirurgie orthopédique',
   'Compte rendu opératoire de Chirurgie orthopédique',
   "Compte rendu d'hospitalisation du service de Chirurgie orthopédique"],
  'text': ["Compte rendu de consultation de Chirurgie orthopédique\n\nLe 24/06/25\n\nMon cher Confrère,\n\nJe vois ce jour en consultation Mme Gérard Danard, âgée de 79 ans, patiente suivie par mes confrères rhumatologues

In [5]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .enableHiveSupport()
    .appName("cu2")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .getOrCreate()
)
sql = spark.sql


# 1. Création du DataFrame initial à partir de la liste
df_raw = spark.createDataFrame(coding_patients)

# 2. Extraction et transformation
# On accède aux champs imbriqués et on sélectionne le premier élément des listes
df = df_raw.select(
    #F.col("documents.text").getItem(0).alias("note_text"),µ
    F.explode("documents.text").alias("note_text"),
    F.col("structured_abstract.primary_diagnosis.code").getItem(0).alias("dp")
    F.col("structured_abstract.primary_diagnosis.code").getItem(0).alias("dp")
)

# Affichage du résultat
df.show(truncate=50)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/20 13:32:42 WARN Utils: Service 'org.apache.spark.network.netty.NettyBlockTransferService' could not bind on port 23527. Attempting port 23528.
/export/home/ldedieu/CU2/encoder-baseline/.venv/lib/python3.12/site-packages/pyspark/sql/session.py:439: UserWarning: inferring schema from dict is deprecated,please use pyspark.sql.Row instead
  warnings.warn("inferring schema from dict is deprecated,"
[Stage 3:=====================================================>   (47 + 3) / 50]

+--------------------------------------------------+-----+
|                                         note_text|   dp|
+--------------------------------------------------+-----+
|Compte rendu de consultation de Chirurgie ortho...|M6534|
|Compte rendu opératoire de Chirurgie orthopédiq...|M6534|
|Compte rendu d'hospitalisation du service de Ch...|M6534|
|Compte rendu de consultation de Chirurgie ortho...| S643|
|Compte rendu opératoire de Chirurgie orthopédiq...| S643|
|Compte rendu d'hospitalisation du service de Ch...| S643|
|Compte rendu de consultation de Chirurgie ortho...|M7747|
|Compte rendu opératoire de Chirurgie orthopédiq...|M7747|
|Compte rendu d'hospitalisation du service de Ch...|M7747|
|Compte rendu de consultation de Chirurgie ortho...|M2323|
|Compte rendu opératoire de Chirurgie orthopédiq...|M2323|
|Compte rendu d'hospitalisation du service de Ch...|M2323|
|Compte rendu de consultation de Chirurgie ortho...|S8280|
|Compte rendu opératoire de Chirurgie orthopédiq...|S828

In [6]:
df.count()

864

In [7]:
df = df.filter(
    (F.col("dp").isNotNull()) & (F.col("dp") != "")
)

In [8]:
df.count()

861

In [16]:
labels_dp = df.select("dp").distinct().rdd.flatMap(lambda x: x).collect()
print(len(labels_dp))

[Stage 14:==================================================>(1986 + 14) / 2000]

129


In [17]:
print(labels_dp)

['K429', 'S431', 'N201', 'M2537', 'S6261', 'N132', 'M2320', 'M7712', 'E6605', 'Z452', 'S661', 'S8280', 'C61', 'K811', 'D351', 'M674', 'N508', 'K642', 'M169', 'K801', 'D180', 'M8098', 'D161', 'M2117', 'M7134', 'K358', 'S6200', 'N47', 'N133', 'N311', 'S460', 'M2556', 'N131', 'M2323', 'M6534', 'A6308', 'M5456', 'S6230', 'N288', 'N480', 'M201', 'K458', 'M2334', 'D414', 'N137', 'G561', 'N130', 'Q556', 'M2324', 'G576', 'M8404', 'L059', 'N318', 'N428', 'K644', 'M2423', 'M0094', 'L030', 'M220', 'N200', 'D171', 'R32', 'M2351', 'D212', 'M7954', 'K402', 'K439', 'M7747', 'M1901', 'S618', 'N135', 'M202', 'S668', 'N432', 'K610', 'K603', 'K802', 'K824', 'S6260', 'M235', 'M4787', 'S610', 'K409', 'S611', 'M7032', 'N328', 'D303', 'M6544', 'M205', 'S663', 'M720', 'K432', 'K800', 'L720', 'N433', 'M233', 'S430', 'N310', 'D293', 'Q530', 'M171', 'N210', 'M1907', 'K818', 'N320', 'C773', 'S3200', 'K648', 'L050', 'K643', 'S8260', 'S860', 'M2322', 'M6504', 'K419', 'M7957', 'K810', 'K641', 'M751', 'M6584', 'K649'

In [18]:
label_counts_dp = (
    df.groupBy("dp")
      .agg(F.count("*").alias("count"))
      .orderBy(F.desc("count"))
)
print(label_counts_dp.count())
label_counts_dp.show()

129


[Stage 22:=========================================>         (1640 + 23) / 2000]

+-----+-----+
|   dp|count|
+-----+-----+
| K409|   51|
| K429|   36|
| N200|   33|
| K801|   30|
| K432|   27|
| K402|   21|
| L059|   21|
| N201|   18|
| K802|   18|
| K439|   18|
| Z452|   18|
| K811|   18|
| S661|   15|
|M6504|   15|
|M7954|   12|
| M751|   12|
|M6584|   12|
| K649|   12|
|S6230|    9|
| K610|    9|
+-----+-----+
only showing top 20 rows



In [19]:
from pyspark.sql import functions as F

def save_df_to_parquet(df, filename, num_partitions):
    df = df.withColumn("note_id", F.monotonically_increasing_id())

    df_to_save = df.select(
        "note_id",
        F.col("note_text"),
        F.col("dp")
    )

    df_to_save.repartition(num_partitions).write.mode("overwrite").parquet(filename)


save_df_to_parquet(df, "CU2/encoder_baseline/PARHAF", num_partitions=1)


In [20]:
import subprocess
user="ldedieu"

subprocess.run(
    ["mkdir", "-p", f"/data/scratch/{user}/CU2/encoder_baseline/PARHAF"],
    check=True
)

subprocess.run(
    [
        "hdfs", "dfs", "-copyToLocal", "-f",
        #f"ia_codage/dataset_800k/{strat}_{n_class_dp}class_v2/*",
        #f"/data/scratch/{user}/ia_codage/DP_DR/{strat}_{n_class_dp}class_v2/"
        f"CU2/encoder_baseline/PARHAF/*",
        f"/data/scratch/{user}/CU2/encoder_baseline/PARHAF/"
    ],
    check=True
)

CompletedProcess(args=['hdfs', 'dfs', '-copyToLocal', '-f', 'CU2/encoder_baseline/PARHAF/*', '/data/scratch/ldedieu/CU2/encoder_baseline/PARHAF/'], returncode=0)

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 47874)
Traceback (most recent call last):
  File "/export/home/ldedieu/.user_conda/miniconda/envs/py312/lib/python3.12/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/export/home/ldedieu/.user_conda/miniconda/envs/py312/lib/python3.12/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
  File "/export/home/ldedieu/.user_conda/miniconda/envs/py312/lib/python3.12/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/export/home/ldedieu/.user_conda/miniconda/envs/py312/lib/python3.12/socketserver.py", line 766, in __init__
    self.handle()
  File "/export/home/ldedieu/CU2/encoder-baseline/.venv/lib/python3.12/site-packages/pyspark/accumulators.py", line 269, in handle
    poll(accum_updates)
  File "/export/home/l